# TD · Classification non supervisée (version étudiant)
**Master 1 Monnaie Banque Finance · Analyse de données · durée : 1 h**

**Données :** le fichier `clients_banque.xlsx` décrit les **2 000 clients** d'une banque de détail. Chaque client est un individu, décrit par 9 variables de comportement : âge, revenus, épargne, crédit immobilier, crédit à la consommation, nombre de produits détenus, paiements par carte, connexions à l'application mobile et jours passés à découvert. Deux autres variables, l'ancienneté et la survenue d'un défaut de paiement dans les 12 mois, servent de **variables illustratives** : elles ne participent pas au calcul des classes.

*Les données sont simulées : aucune banque ne peut diffuser les données de ses clients (secret bancaire, RGPD), mais les ordres de grandeur sont réalistes.*

**Question :** combien de types de clients la banque a-t-elle, comment les reconnaître, et qu'en faire ?

| Partie | Question centrale |
|---|---|
| 1 | Pourquoi segmenter, et comment préparer les données ? |
| 2 | Que fait k-means, et combien de classes garder ? |
| 3 | Qui sont les clients de chaque classe ? |
| 4 | Une seconde méthode (la CAH), et à quoi sert la segmentation ? |

Les cellules contenant `...` sont à compléter. Les questions suivies d'une cellule « Votre réponse » appellent une **réponse rédigée**.

> Placez `clients_banque.xlsx` dans le même dossier que ce notebook. Si `read_excel` réclame `openpyxl` : `pip install openpyxl`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

pd.set_option("display.precision", 2)
clients = pd.read_excel("clients_banque.xlsx", sheet_name="clients", index_col="id_client")
actives = ["age", "revenu", "epargne", "credit_immo", "credit_conso", "nb_produits",
           "paiements_carte", "connexions_appli", "jours_debiteur"]
X = clients[actives]
n, p = X.shape
print(n, "clients |", p, "variables actives | taux de défaut moyen :", round(100 * clients["defaut_12m"].mean(), 2), "%")
X.describe().T

*Unités : revenu en € par mois ; épargne et crédits en milliers d'euros (capital restant dû pour les crédits) ; paiements et connexions par mois ; jours à découvert sur les 12 derniers mois. L'onglet `dictionnaire` du fichier détaille chaque variable.*

### Informations et fonctions fournies
Les numéros de classe que donne k-means sont **arbitraires** (ils dépendent du tirage initial). La fonction `renumeroter` les ordonne selon l'épargne moyenne de la classe (classe 1 = la moins épargnante), pour que tout le monde obtienne les mêmes numéros. Les autres fonctions tracent les graphiques. Exécutez la cellule.

In [ ]:
COULEURS = ["#A9501A", "#2F5DA8", "#3E7C59", "#7B5EA7", "#C08A1E", "#16233A"]
MARQUEURS = "osD^vP"

def renumeroter(etiquettes, ref):
    # classes numérotées 1, 2, 3, etc. par ordre croissant de la moyenne de ref
    s = pd.Series(np.asarray(etiquettes), index=ref.index)
    ordre = ref.groupby(s.values).mean().sort_values().index
    return s.map({c: i + 1 for i, c in enumerate(ordre)}).rename("classe")

def histogrammes(df):
    fig, axes = plt.subplots(3, 3, figsize=(11, 8))
    for ax, col in zip(axes.ravel(), df.columns):
        ax.hist(df[col], bins=40, color="#2F5DA8")
        ax.set_title(col, fontsize=10)
    fig.suptitle("Distribution des 9 variables actives"); plt.tight_layout(); plt.show()

def courbes_k(res):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.plot(res.index, res["inertie intra"], "o-", color="#16233A")
    a1.set_ylabel("inertie intra (par client)"); a1.set_title("Critère du coude")
    a2.bar(res.index, res["silhouette"], color="#2F5DA8")
    a2.set_ylabel("silhouette moyenne"); a2.set_title("Silhouette")
    for a in (a1, a2):
        a.set_xticks(res.index); a.set_xlabel("nombre de classes k")
    plt.tight_layout(); plt.show()

def profil(Z, classe, variables):
    classe = np.asarray(classe)
    m = pd.DataFrame(Z, columns=variables).groupby(classe).mean()
    fig, ax = plt.subplots(figsize=(11, 0.6 * len(m) + 2))
    im = ax.imshow(m.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
    ax.set_xticks(range(len(variables))); ax.set_xticklabels(variables, rotation=35, ha="right")
    ax.set_yticks(range(len(m))); ax.set_yticklabels([f"classe {c} ({(classe == c).sum()} clients)" for c in m.index])
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, f"{m.iloc[i, j]:+.1f}", ha="center", va="center", fontsize=9)
    plt.colorbar(im, label="écart à la moyenne (en écarts-types)")
    ax.set_title("Profil des classes"); plt.tight_layout(); plt.show()

def plan_classes(F, classe, a=0, b=1, titre="Clients dans le plan de l'ACP"):
    classe = np.asarray(classe)
    fig, ax = plt.subplots(figsize=(9, 7))
    for k, c in enumerate(np.unique(classe)):
        m = classe == c
        ax.scatter(F[m, a], F[m, b], s=12, alpha=0.45, marker=MARQUEURS[k], color=COULEURS[k], label=f"classe {c}")
    for k, c in enumerate(np.unique(classe)):
        m = classe == c
        ax.annotate(str(c), (F[m, a].mean(), F[m, b].mean()), fontsize=15, weight="bold", ha="center", va="center",
                    bbox=dict(boxstyle="circle", fc="white", ec=COULEURS[k], lw=2))
    ax.axhline(0, color="grey", lw=0.8); ax.axvline(0, color="grey", lw=0.8)
    ax.set_xlabel(f"F{a+1}"); ax.set_ylabel(f"F{b+1}"); ax.legend(); ax.set_title(titre); plt.show()

def dendro(arbre, coupe=None, p=30):
    fig, ax = plt.subplots(figsize=(11, 4.5))
    seuil = (arbre[-coupe, 2] + arbre[-(coupe - 1), 2]) / 2 if coupe else 0
    dendrogram(arbre, truncate_mode="lastp", p=p, color_threshold=seuil, above_threshold_color="#9AA8BA",
               leaf_font_size=8, ax=ax)
    if coupe:
        ax.axhline(seuil, ls="--", color="#A9501A", label=f"coupe en {coupe} classes"); ax.legend()
    ax.set_ylabel("hauteur de fusion"); ax.set_xlabel("effectif des branches")
    ax.set_title(f"Dendrogramme de la CAH de Ward ({p} dernières branches)"); plt.show()

## Partie 1 · Préparer les données

**Q1.** Affichez les histogrammes (code fourni). Quelles variables sont très **asymétriques** (beaucoup de petites valeurs, quelques très grandes) ? Pourquoi est-ce gênant pour une méthode fondée sur des distances ?

La cellule passe ensuite ces variables au logarithme : `np.log1p(x)` calcule log(1 + x), ce qui accepte les zéros. Vérifiez l'effet sur le coefficient d'asymétrie (`skew` : 0 pour une distribution symétrique).

In [ ]:
histogrammes(X)
asym = ["revenu", "epargne", "credit_immo", "credit_conso", "jours_debiteur"]
Xl = X.copy()
Xl[asym] = np.log1p(X[asym])          # log(1 + x) : accepte les zéros
pd.DataFrame({"asymétrie avant": X.skew(), "après log": Xl.skew()})

**Votre réponse :** …

## Partie 2 · Les k-means

**Q2.** Le directeur marketing est convaincu que la banque a « quatre types de clients ». Centrez-réduisez `Xl`, puis ajustez un k-means à 4 classes (complétez les deux lignes). Combien de clients compte chaque classe ?

In [ ]:
# Q2
scaler = ...                                                     # StandardScaler ajusté sur Xl
Z = ...                                                          # données centrées réduites
km4 = ...                                                        # KMeans à 4 classes ajusté sur Z
classe4 = renumeroter(km4.labels_, Xl["epargne"])
classe4.value_counts().sort_index()

**Q3. Le lien avec le cours (un peu de maths).** Exécutez les trois vérifications. Pour chacune, écrivez en une phrase ce qu'elle signifie concrètement.

*Attention : `km4.inertia_` est une somme de carrés sur les 2 000 clients. On la divise par n pour retrouver la convention du cours (inertie totale = nombre de variables).*

In [ ]:
G = pd.DataFrame(Z).groupby(classe4.values).mean()                 # centres des 4 classes
somme_totale = (Z ** 2).sum()
somme_inter = sum((classe4 == c).sum() * (G.loc[c] ** 2).sum() for c in G.index)
print("1) Inertie totale :", round(somme_totale / n, 2), " / nombre de variables :", p)
print("2) Inertie intra :", round(km4.inertia_ / n, 2), " + inertie inter :", round(somme_inter / n, 2),
      " =", round((km4.inertia_ + somme_inter) / n, 2))
print("3) Part de l'inertie entre les classes (R²) :", round(somme_inter / somme_totale, 3))

**Votre réponse :** …

**Q4. Combien de classes ?** Complétez la boucle qui calcule, pour k = 2 à 10, l'inertie intra (par client) et la silhouette moyenne, puis tracez les deux courbes avec `courbes_k`. Où se trouve le coude ? Que dit la silhouette ? Les critères sont-ils plus nets que pour les entreprises du cours ? Le directeur marketing avait-il raison ?

In [ ]:
# Q4
resultats = []
for k in range(2, 11):
    km = ...                                   # KMeans à k classes ajusté sur Z
    resultats.append((k, ..., ...))            # inertie intra par client, silhouette moyenne
resultats = pd.DataFrame(resultats, columns=["k", "inertie intra", "silhouette"]).set_index("k")
courbes_k(resultats)
resultats.assign(baisse=-resultats["inertie intra"].diff()).round(3)

**Votre réponse :** …

**Q5.** Le modèle final à 5 classes est ajusté ci-dessous. Testez sa **stabilité** : relancez k-means avec une seule initialisation (`n_init=1`) et `random_state` de 0 à 5 ; comparez l'inertie et l'accord avec le modèle final (indice de Rand ajusté, ARI : 1 = exactement les mêmes classes, 0 = accord dû au hasard). Que constatez-vous ? Pourquoi scikit-learn relance-t-il l'algorithme plusieurs fois (`n_init`) ?

In [ ]:
km5 = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Z)
classe = renumeroter(km5.labels_, Xl["epargne"])
print("modèle final : inertie intra =", round(km5.inertia_ / n, 3))
# Q5
for graine in range(6):
    essai = ...                                # KMeans à 5 classes, n_init=1, random_state=graine
    print("graine", graine, "| inertie intra =", round(essai.inertia_ / n, 3),
          "| accord avec le modèle final (ARI) =", round(adjusted_rand_score(km5.labels_, essai.labels_), 2))

**Votre réponse :** …

## Partie 3 · Interpréter les classes

### 3.1 Le portrait des classes

**Q6.** Tracez le profil des classes (`profil`) : chaque case donne l'écart entre la moyenne de la classe et la moyenne générale, **en écarts-types**. Complétez par les médianes de chaque classe en unités d'origine. **Nommez les 5 classes** et justifiez chaque nom par 2 ou 3 variables.

In [ ]:
# Q6
profil(Z, classe, actives)
...                               # médianes de chaque variable par classe, en unités d'origine

**Votre réponse :** …

**Q7. Les variables illustratives.** Calculez, pour chaque classe, l'effectif, le taux de défaut à 12 mois, l'ancienneté moyenne et la **part de tous les défauts** de la banque qu'elle représente (complétez la dernière ligne). Quelle classe concentre le risque ? Ces deux variables ont-elles servi à former les classes ? Quel usage la banque peut-elle en faire ?

In [ ]:
# Q7
illus = clients.groupby(classe).agg(effectif=("defaut_12m", "size"), taux_defaut_pct=("defaut_12m", "mean"),
                                    anciennete=("anciennete", "mean"))
illus["taux_defaut_pct"] *= 100
illus["part_des_defauts_pct"] = ...
illus.round(1)

**Votre réponse :** …

### 3.2 Les classes sur la carte de l'ACP, et un piège

**Q8. Les classes dans le plan de l'ACP.** L'ACP des données centrées réduites est calculée ci-dessous (comme à la séance précédente). Nommez rapidement F1 et F2 à l'aide des corrélations, puis représentez les clients dans le plan F1–F2, colorés par classe (`plan_classes`). Les classes se séparent-elles ? Lesquelles se chevauchent ?

In [ ]:
acp = PCA().fit(Z)
lam = acp.explained_variance_ * (n - 1) / n                         # convention 1/n du cours
signes = np.sign(acp.components_[:, actives.index("epargne")])       # le signe d'un axe est arbitraire :
F = acp.transform(Z) * signes                                        # on oriente chaque axe pour que l'épargne y soit positive
corr = pd.DataFrame(acp.components_.T * np.sqrt(lam) * signes, index=actives,
                    columns=[f"F{k+1}" for k in range(p)])
print("Part d'inertie de F1 et F2 (%) :", (100 * lam[:2] / p).round(1))
display(corr[["F1", "F2"]].sort_values("F1"))
# Q8
...

**Votre réponse :** …

**Q9. Le piège de l'unité.** Relancez k-means à 5 classes sur `Xl` **sans** centrer-réduire, puis comparez avec `classe` (tableau croisé). Que deviennent les clients fragiles (classe 1) ? Pourquoi ? (Aide : regardez les variances de `Xl`.)

In [ ]:
# Q9
km_brut = ...                                  # KMeans à 5 classes sur Xl, sans centrer-réduire
classe_brut = renumeroter(km_brut.labels_, Xl["epargne"])
display(pd.crosstab(classe_brut, classe, rownames=["sans réduction"], colnames=["avec réduction"]))
Xl.var().round(2).sort_values()

**Votre réponse :** …

## Partie 4 · La CAH, puis l'utilisation des classes

### 4.1 Une seconde méthode : la CAH

**Q10. La CAH de Ward.** Calculez l'arbre (complétez), puis tracez le dendrogramme (`dendro`, les 30 dernières branches). Où couperiez-vous l'arbre ? Coupez-le en 5 classes et comparez avec k-means (tableau croisé et ARI). Les deux méthodes s'accordent-elles ?

*Pour Ward, la hauteur h d'une fusion est liée à l'inertie perdue : h²/2 est l'augmentation de l'inertie intra (en somme de carrés) provoquée par cette fusion.*

In [ ]:
# Q10
arbre = ...                                    # CAH de Ward sur Z
print("hauteurs des 6 dernières fusions :", arbre[-6:, 2].round(1))
dendro(arbre, coupe=5)
classe_cah = renumeroter(..., Xl["epargne"])   # couper l'arbre en 5 classes
display(pd.crosstab(classe_cah, classe, rownames=["CAH"], colnames=["k-means"]))
print("Clients classés pareil :", round((classe_cah == classe).mean(), 3),
      "| ARI :", round(adjusted_rand_score(classe_cah, classe), 2))

**Votre réponse :** …

### 4.2 Classer de nouveaux clients

**Q11.** L'onglet `nouveaux_clients` contient 4 clients arrivés ce mois-ci. Préparez-les **exactement comme les données d'origine** (passage au log, puis le `scaler` déjà ajusté), puis prédisez leur classe avec `km5.predict`. Le client N4 est-il facile à classer ? (Aide : `km5.transform` donne la distance de chaque client à chaque centre.) Que se passerait-il si l'on centrait-réduisait un nouveau client seul, avec un nouveau `StandardScaler` ?

In [ ]:
nouveaux = pd.read_excel("clients_banque.xlsx", sheet_name="nouveaux_clients", index_col="id_client")
display(nouveaux)
Nl = nouveaux[actives].copy()
Nl[asym] = np.log1p(Nl[asym])
# Q11
Zn = ...                          # même transformation que pour Z (scaler déjà ajusté : ne pas le réajuster)
codes = ...                       # classe prédite par km5
numero = dict(zip(km5.labels_, classe))          # code donné par k-means -> numéro renuméroté
print("classes prédites :", [int(numero[c]) for c in codes])
distances = pd.DataFrame(km5.transform(Zn), index=nouveaux.index).rename(columns=numero).sort_index(axis=1)
distances.round(2)

In [ ]:
# Q11 (suite) : le piège du nouveau scaler
seul = StandardScaler().fit_transform(Nl.loc[["N3"]])
print("N3 centré-réduit seul :", seul.round(2), "-> classe", int(numero[km5.predict(seul)[0]]))

**Votre réponse :** …

### 4.3 Communiquer les résultats

**Q12.** Rédigez en **5 à 6 lignes** l'encadré d'une note destinée au comité de direction de la banque, qui n'a jamais entendu parler de k-means : ce qu'on a fait, combien de segments et leur poids, ce qui les distingue, où se concentre le risque, une action par segment ou presque, et une limite.

**Votre réponse :** …